## Setup and Dependency Installation

Install the required Python packages for running the notebook. We use `wandb` for potential logging and `tqdm` for progress bars.

In [ ]:
!pip install wandb tqdm -q


## Drive Mount

Mount Google Drive to access the `RAW_DATA` and store the computed `.pt` cache files in `DATA_ROOT`. This ensures our generated artifacts are preserved beyond the lifespan of the Colab session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Path Configuration

Define the standard repository, data, and raw data paths. Additionally, create the necessary subdirectory structure in `DATA_ROOT` so we have a consistent layout across all notebooks.

In [ ]:
REPO_ROOT = '/content/antenna-gnn'
DATA_ROOT = '/content/drive/MyDrive/antenna_gnn'
RAW_DATA  = '/content/drive/MyDrive/antenna_dataset'

import os
for d in [f'{DATA_ROOT}/artifacts', f'{DATA_ROOT}/checkpoints',
          f'{DATA_ROOT}/figures',   f'{DATA_ROOT}/splits',
          f'{DATA_ROOT}/data/processed', f'{REPO_ROOT}/notebooks']:
    os.makedirs(d, exist_ok=True)
print(f'Drive mounted. DATA_ROOT={DATA_ROOT}')


## Multi-Grid Cache Generation

This cell implements a fast parallel loading pattern to assemble the `.mat` datasets into efficiently loadable `.pt` PyTorch cache files for each grid size (25, 35, 45, and 55).

To overcome Google Drive's severe I/O latency when sequentially accessing thousands of files, we:
1. Use `glob` to locate the files on Drive.
2. Execute a parallel `xargs cp` command using 16 workers to bulk-copy the raw `.mat` files to the local Colab disk (`/content/raw_NxN/`).
3. Parse the local files with a `ThreadPoolExecutor` (16 workers) to extract the `patch_pattern`, `S11_dB` spectrum, and `is_functioning` flag.
4. Stack the data into PyTorch tensors and cache them to `DATA_ROOT/artifacts/`.

In [ ]:
import glob
import os
import subprocess
import concurrent.futures
import scipy.io as sio
import torch
import numpy as np
from tqdm.auto import tqdm

RAW_PATHS = {
    25: f'{RAW_DATA}/training dataset/25x25/**/Mat_Files/*.mat',
    35: f'{RAW_DATA}/fine-tuning dataset/35x35/**/Mat_Files/*.mat',
    45: f'{RAW_DATA}/fine-tuning dataset/45x45/**/Mat_Files/*.mat',
    55: f'{RAW_DATA}/fine-tuning dataset/55x55/**/Mat_Files/*.mat',
}

def load_one(path):
    try:
        mat = sio.loadmat(path)
        pattern = mat['patch_pattern'].astype(np.float32)
        s11 = mat['S11_dB'].flatten().astype(np.float32)
        is_functioning = 1 if mat['resonant_freqs'].size > 0 else 0
        return pattern, s11, is_functioning
    except Exception as e:
        return None

for N in [25, 35, 45, 55]:
    print(f'\n=== Processing Grid {N}x{N} ===')
    raw_paths = sorted(glob.glob(RAW_PATHS[N], recursive=True))
    count = len(raw_paths)
    print(f'Found {count} raw .mat files on Drive.')
    
    if count == 0:
        print(f'Skipping {N}x{N} because no files were found.')
        continue
    
    # Create local target directory
    local_dir = f'/content/raw_{N}x{N}'
    os.makedirs(local_dir, exist_ok=True)
    
    # Bulk copy to local disk using parallel python threads
    print('Bulk copying files to local disk...')
    import shutil
    def copy_file(src):
        dst = os.path.join(local_dir, os.path.basename(src))
        shutil.copy(src, dst)
        return dst
        
    with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
        local_paths = list(tqdm(executor.map(copy_file, raw_paths), total=count, desc='Copying files'))
    
    print('Parsing local files...')
    patterns = []
    s11_list = []
    func_list = []
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
        results = list(tqdm(executor.map(load_one, local_paths), total=len(local_paths)))
        
    for r in results:
        if r is not None:
            p, s, f = r
            patterns.append(p)
            s11_list.append(s)
            func_list.append(f)
    
    patterns_t = torch.tensor(np.stack(patterns))
    s11_t = torch.tensor(np.stack(s11_list))
    func_t = torch.tensor(np.array(func_list))
    
    func_pct = 100.0 * func_t.float().mean().item()
    print(f'Processed {len(func_t)} samples. Functioning: {func_pct:.2f}%')
    
    out_path = f'{DATA_ROOT}/artifacts/raw_cache_{N}x{N}.pt'
    torch.save({
        'patterns': patterns_t,
        's11': s11_t,
        'is_functioning': func_t
    }, out_path)
    print(f'Saved cache to {out_path}')


## Cache Verification

Finally, we check the sizes of the generated PyTorch `.pt` cache files on disk to confirm successful creation and understand their upload footprint.

In [ ]:
import os
print('Cache file sizes:')
for N in [25, 35, 45, 55]:
    path = f'{DATA_ROOT}/artifacts/raw_cache_{N}x{N}.pt'
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f'  raw_cache_{N}x{N}.pt: {size_mb:.1f} MB')
    else:
        print(f'  raw_cache_{N}x{N}.pt: Not found')
